# True-Strength — a quantitative teardown 🔬
### Spanning-R² · sign agreement · equity-curve ρ · long/short alpha-vs-beta · White (2000) Reality Check · cost sweep

![Signal: None](https://img.shields.io/badge/Signal-None-c0392b?style=flat-square)
![Tradability: Mirage](https://img.shields.io/badge/Tradability-Mirage-c0392b?style=flat-square)
![Truer?: Busted](https://img.shields.io/badge/Truer-Busted-8b949e?style=flat-square)

The deep companion to the [notebook for the curious](01_for_the_curious.ipynb) — *same seven beats, every claim now carrying its standard error.* We ask whether the TSI carries momentum information not already in MACD+RSI, or is merely the same trade double-smoothed.

> ⚠️ **Not investment advice.** Executed on the offline synthetic universe; the headline real numbers (177-name basket) are quoted from `../docs/results.md` (as-of + fingerprint), produced by `examples/verify_real.py`. References in `../docs/references.md`.
>
> 💡 **The `💡 In plain words` notes** translate each result back into intuition — so this notebook still reads even if you skim the maths. House style in [METHODOLOGY.md](../../../METHODOLOGY.md).

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))           # study root (true_strength/ lives there)
sys.path.insert(0, os.path.abspath("../../.."))      # repo root, for quantlab
%matplotlib inline
import matplotlib.pyplot as plt
plt.rcParams["figure.figsize"] = (9.5, 5.2)
import numpy as np, pandas as pd
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")
from true_strength import data, oscillators as osc, backtest, collinearity

# Offline synthetic universe: trend/cycle names among random walks. This is the CONTROL
# (the collinearity is as high on pure noise as on structure). The real verdict (a liquid
# 177-name basket: NONE / MIRAGE / BUSTED) is in ../docs/results.md via verify_real.py.
frames, truth = data.synthetic_universe(seed=0)
structured = {t.ticker for t in truth}
print(f"{len(frames)} names, {len(structured)} with planted momentum structure")


20 names, 8 with planted momentum structure


## 1–3 · Claim, stakes, protocol

H₁: the TSI carries momentum information not already in MACD+RSI. Null: it is spanned by them, its position and equity curve are collinear, and any standalone Sharpe is the long bias. Pre-registered mirage line: R² ≳ 0.8, sign agreement ≳ 0.9 vs MACD, equity ρ ≳ 0.95. The closing-argument criterion is declared on the **gross** residual Sharpe (information), with net (friction) reported separately.

The control — and what it actually proves. Position agreement is a little higher on planted structure than on noise (the trades line up where there is a trend to agree about), but the **level collinearity and the spanning R² are about as high on pure random walks**: the TSI, the MACD line and the centred RSI are all smoothings of the same one-bar price change, so they co-move on *any* input. The collinearity is mechanical — which sharpens the verdict: a near-identity built into the filter cannot be a distinct signal on any data.

In [2]:
rec = collinearity.structure_recall(frames, truth)
print(rec)
# corr_tsi_macd_noise / spanning_r2_noise are the load-bearing numbers: as high on
# random walks as on structure (and as on the real 177-name panel).

{'agree_structured': 0.846875, 'agree_noise': 0.8158333333333333, 'corr_tsi_macd_structured': 0.9336778502029057, 'corr_tsi_macd_noise': 0.8985903038302523, 'spanning_r2_structured': 0.9121672723804825, 'spanning_r2_noise': 0.8594945884105931}


## 4 · The Teardown

### Same shape? Level collinearity and the spanning-R²

In [3]:
print(collinearity.level_collinearity(frames).round(3).to_string())
print('\nincremental information (TSI on MACD+RSI):', 
      {k: round(v,3) if isinstance(v,float) else v for k,v in collinearity.incremental_information(frames).items()})

          n_names  median_corr    q25    q75
pair                                        
tsi~macd       20       0.9300 0.8960 0.9350
tsi~rsi        20       0.8720 0.8610 0.8890
macd~rsi       20       0.8390 0.8050 0.8620



incremental information (TSI on MACD+RSI): {'median_name_r2': 0.893, 'pooled_r2': 0.88, 'n_names': 20}


### Same position, same equity curve?

In [4]:
print('sign agreement (zero-cross):  ', {k: round(v,3) for k,v in collinearity.sign_agreement(frames,'zero').items()})
print('sign agreement (signal-cross):', {k: round(v,3) for k,v in collinearity.sign_agreement(frames,'signal').items()})
eq = collinearity.equity_collinearity(frames)
print('equity-curve correlation + Sharpes:', {k: round(v,3) for k,v in eq.items()})

sign agreement (zero-cross):   {'all_three': 0.828, 'tsi=macd': 0.986, 'tsi=rsi': 0.835, 'macd=rsi': 0.836}


sign agreement (signal-cross): {'all_three': 0.682, 'tsi=macd': 0.916, 'tsi=rsi': 0.73, 'macd=rsi': 0.719}


equity-curve correlation + Sharpes: {'corr_tsi_macd': 0.971, 'corr_tsi_rsi': 0.674, 'corr_macd_rsi': 0.684, 'sharpe_tsi': 0.12, 'sharpe_macd': 0.018, 'sharpe_rsi': -0.89}


### Alpha vs beta — the long/short timing cut

The standalone long/flat Sharpe is partly the equity risk premium (long ~half the time). Symmetrise to long/short and the unconditional drift cancels, leaving only the oscillator's *timing*. On the real run that collapses TSI 0.63 → −0.42 (same signal-cross rule; the zero-cross long/short book sits at +0.05).

In [5]:
# long/flat vs long/short Sharpe for the TSI crossover on this synthetic universe
lf = backtest.equal_weight_net(frames, lambda c: backtest.tsi_position(c, rule='signal', long_short=False))
ls = backtest.equal_weight_net(frames, lambda c: backtest.tsi_position(c, rule='signal', long_short=True))
print('long/flat  Sharpe:', round(backtest.annualized_sharpe(lf), 3))
print('long/short Sharpe:', round(backtest.annualized_sharpe(ls), 3))

long/flat  Sharpe: -0.367
long/short Sharpe: -1.037


### Reality Check on the 24-variant TSI grid — three nulls, one honest one

White (2000), stationary bootstrap, best-of-grid Sharpe. Run **long/flat** the test embeds the equity drift (every variant is in the market ~half the time), so a tiny p certifies *being long stocks*, not the oscillator. The honest runs strip the beta: score each variant **in excess of equal-weight buy-and-hold** (does the timing beat simply holding?) or **dollar-neutral** (long/short, drift cancels inside the book).

In [6]:
rc_lf = collinearity.reality_check_grid(frames, n_boot=500)
rc_xs = collinearity.reality_check_grid(frames, n_boot=500, excess_of_bh=True)
rc_ls = collinearity.reality_check_grid(frames, n_boot=500, long_short=True)
for tag, rc in [('long/flat ', rc_lf), ('excess B&H', rc_xs), ('lng/short ', rc_ls)]:
    print(tag, {k: (round(v,4) if isinstance(v,float) else v) for k,v in rc.items()})

long/flat  {'n_variants': 24, 'long_short': False, 'benchmark': 'zero', 'best_variant': 'tsi_40_21_13', 'best_sharpe': 0.1331, 'reality_check_pvalue': 0.634}
excess B&H {'n_variants': 24, 'long_short': False, 'benchmark': 'buy_and_hold', 'bh_sharpe': 0.5094, 'best_variant': 'tsi_40_21_13', 'best_sharpe': -0.5908, 'reality_check_pvalue': 0.988}
lng/short  {'n_variants': 24, 'long_short': True, 'benchmark': 'zero', 'best_variant': 'tsi_40_21_13', 'best_sharpe': -0.3348, 'reality_check_pvalue': 0.978}


### Cost sweep

In [7]:
print(collinearity.cost_sweep(frames, rule='signal').round(3).to_string())

          mean_net_bps  sharpe  ann_turnover
cost_bps                                    
0               0.1250  0.0820       17.3160
5              -0.2180 -0.1420       17.3160
10             -0.5620 -0.3670       17.3160
20             -1.2490 -0.8140       17.3160
40             -2.6230 -1.7040       17.3160


### The closing argument — the TSI's residual over MACD+RSI: information vs friction

Regress the TSI out of MACD+RSI (full-sample, the generous steelman) and trade the *residual* — the part the other two can't reproduce. The pre-declared criterion is on the **gross** Sharpe: ≈ 0 means the unique content carries no information, and on the real run that is exactly what it earns (gross Sharpe −0.02, HAC *t* −0.2). The **net** number is a separate, friction statement: the residual flips sign ~4× as often as the level (~72×/yr vs ~16×/yr), so any per-side cost drags a zero-information signal reliably negative (−0.59 net at 10 bps/side) — cost arithmetic, not a tradable anti-signal.

In [8]:
print({k: (round(v,4) if isinstance(v,float) else v)
       for k,v in collinearity.orthogonalised_tsi_edge(frames).items()})

{'n_names': 20, 'residual_gross_sharpe': 0.0503, 'residual_gross_sharpe_se': 0.4194, 'residual_gross_mean_bps': 0.1132, 'residual_gross_hac_t': 0.1201, 'raw_tsi_ls_gross_sharpe': 0.4267, 'residual_net_sharpe': -1.5169, 'residual_net_mean_bps': -3.4125, 'raw_tsi_ls_net_sharpe': 0.1202, 'raw_tsi_ls_net_mean_bps': 0.269, 'residual_ann_turnover': 88.848, 'raw_ann_turnover': 17.298}


## 5–7 · Verdict, tradability, going further

**Signal `NONE`** (spanning R² 0.835, sign agreement 0.994 vs MACD, equity ρ 0.994 — all mechanical to the filters per the control). **Tradability `MIRAGE`** (the crossover's 0.63 was beta: the same rule long/short earns −0.42, the zero-cross book +0.05; in excess of buy-and-hold the best of 24 variants is −0.83 with Reality Check p = 1.00). The orthogonalised residual earns **≈ 0 gross** (−0.02, the pre-declared criterion) and −0.59 net at 10 bps/side — friction, not anti-signal. **'Truer'? `BUSTED`.** Next: extend to the wider oscillator zoo (Stochastic, CCI, %R). Numbers + fingerprint in `../docs/results.md`.